In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
OUTPUT = TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y = CONFIG['last_year']

In [ ]:
ranks = pd.read_csv(OUTPUT/'rank_correlations.csv')
distributions = pd.read_csv(OUTPUT/'distribution_summary.csv')
distances = pd.read_csv(OUTPUT/'distribution_distances.csv')
moments = pd.read_csv(OUTPUT/'annual_moments.csv')
qq_summary = pd.read_csv(OUTPUT/'window_qq_summary.csv')

In [ ]:
# common late window; lag-3 and lag-5 pairs fully generated for AR(6,6)
annual = ranks.loc[ranks.measure.eq('annual')]
empirical = annual.loc[annual.source.eq('empirical_complete')]
rows = []
for source in ['empirical_complete'] + CONFIG['models']:
    selected = annual.loc[annual.source.eq(source)]
    row = {'source':source, 'n':int(selected.n.iloc[0]),
           'generated_from_age':6 if source=='ar6_6' else 1 if source!='empirical_complete' else np.nan,
           'fit_cohort':'complete careers' if source=='se_hurdle_s' else 'available observations' if source!='empirical_complete' else 'complete careers'}
    for reference in ['age_0','lag_3','lag_5']:
        row[f'{reference}_rho20'] = selected.loc[selected.reference.eq(reference)&selected.age.eq(Y),'rho'].iloc[0]
        late = selected.loc[selected.reference.eq(reference)&selected.age.between(11,Y)]
        matched = late.merge(empirical.loc[empirical.reference.eq(reference),['age','rho']],on='age',suffixes=('_sim','_emp'))
        row[f'{reference}_rmse11_20'] = np.sqrt(np.mean((matched.rho_sim-matched.rho_emp)**2))
    totals = distributions.loc[distributions.source.eq(source)&distributions.window.eq('career_total')].iloc[0]
    last_four = distributions.loc[distributions.source.eq(source)&distributions.window.eq('last_four')].iloc[0]
    last_shape = qq_summary.loc[qq_summary.source.eq(source)&qq_summary.window.eq('last_four')].iloc[0]
    career_shape = qq_summary.loc[qq_summary.source.eq(source)&qq_summary.window.eq('career_total')].iloc[0]
    generated = distributions.loc[distributions.source.eq(source)&distributions.window.eq('generated_common')].iloc[0]
    final = moments.loc[moments.source.eq(source)&moments.age.eq(Y)].iloc[0]
    row.update({'lognormality_window':'last_four', 'last_four_start_age':Y-3, 'last_four_end_age':Y,
                'mean_last_four_total':last_four.mean_total, 'gini_last_four_total':last_four.gini_total,
                'last_four_n_positive':int(last_shape.n_positive),
                'last_four_zero_fraction':1-last_shape.n_positive/last_shape.n_total,
                'last_four_qq_r':last_shape.qq_r, 'last_four_qq_r2':last_shape.qq_r2,
                'last_four_ks_D':last_shape.ks_D, 'career_total_qq_r2':career_shape.qq_r2,
                'mean_career_total':totals.mean_total, 'gini_career_total':totals.gini_total,
                'mean_total6_20':generated.mean_total, 'gini_total6_20':generated.gini_total,
                'final_mean':final['mean'], 'final_zero_fraction':final.zero_fraction})
    if source!='empirical_complete':
        distance = distances.loc[distances.source.eq(source)&distances.diagnostic.eq('generated_common')].iloc[0]
        row['total6_20_ks_D'] = distance.ks_D
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT/'model_comparison.csv',index=False)
display(comparison)

In [ ]:
# final-four-year sums are primary; whole-career sums retained as a supplement
lognormality = qq_summary.loc[qq_summary.source.isin(['empirical_complete']+CONFIG['models'])
                             & qq_summary.window.isin(['last_four','career_total'])].copy()
lognormality['role'] = lognormality.window.map({'last_four':'primary','career_total':'supplement'})
lognormality['start_age'] = np.where(lognormality.window.eq('last_four'),Y-3,0)
lognormality['end_age'] = Y
lognormality['n_zero'] = lognormality.n_total-lognormality.n_positive
lognormality['zero_fraction'] = lognormality.n_zero/lognormality.n_total
lognormality = lognormality.merge(distributions[['source','window','mean_total','gini_total']],
                                  on=['source','window'])
columns = ['source','window','role','start_age','end_age','n_total','n_positive','n_zero','zero_fraction',
           'mean_total','gini_total','loc','fit_scale','qq_r','qq_r2','ks_D','prefix_included']
lognormality = lognormality[columns].sort_values(['start_age','source'], ascending=[False,True])
lognormality.to_csv(OUTPUT/'lognormality_summary.csv',index=False)
display(lognormality.loc[lognormality.role.eq('primary')])